In [0]:
def ensure_logging_table(catalog, schema_name):
    """
    Creates ETL run log table if it doesn't exist with comprehensive metrics.
    
    Args:
        catalog: Catalog name
        schema_name: Schema name for the log table
    """
    full_table_name = f"{catalog}.{schema_name}.etl_run_log"
    
    try:
        table_exists = spark.catalog.tableExists(full_table_name)
        
        if not table_exists:
            logger.info(f"Creating comprehensive logging table: {full_table_name}")
            spark.sql(f"""
                CREATE TABLE IF NOT EXISTS {full_table_name} (
                    run_id STRING NOT NULL COMMENT 'Unique identifier for the ETL run',
                    table_name STRING NOT NULL COMMENT 'Name of the table being processed',
                    log_datetime TIMESTAMP NOT NULL COMMENT 'Timestamp when the run started',
                    status STRING NOT NULL COMMENT 'Run status: SUCCESS, FAILED, or SKIPPED',
                    load_type STRING COMMENT 'Type of load: merge, overwrite, skip, or error',
                    error_message STRING COMMENT 'Error details if failed',
                    
                    -- Timing metrics
                    duration_seconds DOUBLE COMMENT 'Total execution time in seconds',
                    duration_minutes DOUBLE COMMENT 'Total execution time in minutes',
                    read_duration_seconds DOUBLE COMMENT 'Time spent reading source data',
                    write_duration_seconds DOUBLE COMMENT 'Time spent writing to target',
                    
                    -- Row count metrics
                    source_row_count BIGINT COMMENT 'Number of rows in source query',
                    target_row_count BIGINT COMMENT 'Number of rows in target after operation',
                    inserted INT COMMENT 'Number of rows inserted',
                    updated INT COMMENT 'Number of rows updated',
                    deleted INT COMMENT 'Number of rows deleted',
                    copied INT COMMENT 'Number of rows copied (unchanged in merge)',
                    
                    -- Configuration
                    incremental_interval INT COMMENT 'Incremental interval in days (if applicable)',
                    was_forced_full_load BOOLEAN COMMENT 'Whether this was forced to full load due to missing table'
                )
                USING delta
                COMMENT 'Comprehensive ETL run log for tracking pipeline execution and performance'
            """)
            logger.info(f"Successfully created {full_table_name}")
        else:
            logger.info(f"Logging table already exists: {full_table_name}")
            
    except Exception as e:
        logger.error(f"Failed to ensure logging table: {e}")
        raise